# Qiskit tomography notebook for BHZ, SSH, and BBH effective quartets

This notebook runs two-qubit state tomography, and optional process tomography, for the representative effective loops used in the paper:

- **BHZ ribbon**: `CT`, `CB`, `C_plus`, `C_minus`
- **spinful SSH chain**: `CL`, `CR`, `Cdiag`, `Canti`
- **BBH corner quartet**: `Cx`, `Cy`, `Cdiag`, `Canti`

The workflow builds the circuits, executes the selected backend, reconstructs the output density matrices, optionally reconstructs the full two-qubit processes, and saves JSON/CSV summaries together with PNG figures.

Backend options:

- `"local_aer"`
- an IBM Quantum backend name such as `"ibm_brisbane"`
- `"analytic_reference"` for reference evaluation without Qiskit

`SCAN_POINTS` controls the density of the smooth loop scans used for the overview plots.


## Optional first-run install

Uncomment and run this cell on a fresh environment that does not yet have the Qiskit stack installed.


In [ ]:
# %pip install -q "qiskit>=1.2" "qiskit-aer>=0.15" "qiskit-ibm-runtime>=0.31" "matplotlib>=3.8" "scipy>=1.11" "pandas>=2.2" "notebook>=7.0"


## Imports and catalog

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

sys.path.insert(0, str(Path.cwd()))

from quartet_tomography_all_models import (
    ExecutionConfig,
    catalog_dataframe,
    load_overview_dataframe,
    run_suite,
)

available = catalog_dataframe()
display(available)


## Configuration

- Set `BACKEND_NAME = "local_aer"` for Qiskit Aer.
- To run on IBM Quantum hardware, change only `BACKEND_NAME`, for example to `"ibm_brisbane"`.
- `SCAN_POINTS` controls the density of the smooth classical scans used to generate the plots.


In [ ]:
OUTPUT_DIR = Path("quartet_tomography_notebook_outputs")

BACKEND_NAME = "local_aer"
# Examples:
# BACKEND_NAME = "ibm_brisbane"
# BACKEND_NAME = "analytic_reference"

MODELS = ("BHZ", "SSH", "BBH")
SELECTED_LOOPS = None  # or e.g. {"BHZ": ["CT", "C_plus", "C_minus"]}
SHOTS = 4096
WITH_PROCESS_TOMOGRAPHY = True
SCAN_POINTS = 301

IBM_TOKEN = None      # optional if already saved with QiskitRuntimeService.save_account(...)
IBM_INSTANCE = None   # optional / recommended for hardware
IBM_CHANNEL = "ibm_quantum_platform"

config = ExecutionConfig(
    backend_name=BACKEND_NAME,
    models=MODELS,
    shots=SHOTS,
    output_dir=str(OUTPUT_DIR),
    with_process_tomography=WITH_PROCESS_TOMOGRAPHY,
    scan_points=SCAN_POINTS,
    token=IBM_TOKEN,
    instance=IBM_INSTANCE,
    channel=IBM_CHANNEL,
    selected_loops=SELECTED_LOOPS,
)

config


## Run the full suite

This cell executes the full pipeline:

1. build the circuits,
2. run the selected backend,
3. reconstruct output density matrices,
4. optionally reconstruct the full processes,
5. save raw data and figures.


In [ ]:
summary = run_suite(config)
overview_df = load_overview_dataframe(OUTPUT_DIR)

display(
    overview_df.style.format(
        {
            "paper_dloc": "{:.3f}",
            "witness_concurrence": "{:.3f}",
            "witness_state_fidelity": "{:.4f}",
            "estimated_dloc": "{:.3f}",
            "ideal_dloc": "{:.3f}",
            "unitary_overlap": "{:.4f}",
            "schmidt_s1": "{:.3f}",
            "schmidt_s2": "{:.3f}",
            "schmidt_s3": "{:.3f}",
            "schmidt_s4": "{:.3f}",
        },
        na_rep="-",
    )
)

summary["artifacts"]

## Key overview plots

In [ ]:
overview_keys = [
    "overview_dloc_plot",
    "overview_concurrence_plot",
    "overview_operator_schmidt_plot",
]
for key in overview_keys:
    if key in summary["artifacts"]:
        display(Markdown(f"### {key}"))
        display(Image(filename=summary["artifacts"][key]))


## Per-model tomography and smooth scans

In [ ]:
for model in MODELS:
    display(Markdown(f"## {model}"))
    for suffix in [
        "selected_correlators_plot",
        "dloc_scan_plot",
        "concurrence_scan_plot",
    ]:
        key = f"{model.lower()}_{suffix}"
        if key in summary["artifacts"]:
            display(Markdown(f"### {key}"))
            display(Image(filename=summary["artifacts"][key]))


## Saved artifacts on disk

In [ ]:
artifact_paths = pd.DataFrame(
    [{"artifact": key, "path": value} for key, value in summary["artifacts"].items()]
)
display(artifact_paths)

print("Output directory:", OUTPUT_DIR.resolve())
print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)


## Minimal hardware rerun template

To move from a local simulator to real hardware, keep the notebook unchanged and change only the backend name:

```python
config = ExecutionConfig(
    backend_name="ibm_brisbane",
    models=("BHZ", "SSH", "BBH"),
    shots=2048,
    output_dir="quartet_tomography_hw_outputs",
    with_process_tomography=True,
    scan_points=301,
    token=None,
    instance=None,
    channel="ibm_quantum_platform",
)
summary = run_suite(config)
```

For an initial hardware test, it is often convenient to reduce `SHOTS` or set `WITH_PROCESS_TOMOGRAPHY = False`, and then re-enable the full process reconstruction after the first successful run.
